In [1]:
import pandas as pd,numpy as np,json
from sklearn.metrics.pairwise import cosine_distances
import getpass,os
from langchain.chat_models import init_chat_model
from ai_patterns_mining import parse_json_safe,Config
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.store.memory import InMemoryStore
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from time import time
from langchain.tools import tool
from pydantic import BaseModel, Field
import seaborn as sns
import matplotlib.pyplot as plt
from langchain.agents.structured_output import ToolStrategy
from tqdm import tqdm

/home/hasinthaka/Documents/Projects/AI/Pattern Mining/pipeline/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
predicted_clusters = pd.read_json("/home/hasinthaka/Documents/Projects/AI/Pattern Mining/pipeline/notebooks/result/pattern_classification_verification_results_NN_v2_list.json")

In [5]:
predicted_clusters.head()

,code file,predicted labels,llm predicted high level pattern,llm predicted l2 pattern,confidence score,gemini verification explaination,code summary
0,https://github.com/HasinthakaPiyumal/AI-Patter...,LLM based Multimodal Generative Prompting: 0.4868,LLM based Multimodal Generative Prompting,3D Prompting,9.0,The code processes multimodal data (Lidar poin...,This code implements a data pipeline for 3D ob...
1,https://github.com/HasinthakaPiyumal/AI-Patter...,LLM based Multimodal Generative Prompting: 0.2496,None,None,1.0,The code implements 3D object pose estimation ...,This code implements a 3D object pose estimati...
2,https://github.com/HasinthakaPiyumal/AI-Patter...,LLM based Multimodal Generative Prompting: 0.5620,LLM based Multimodal Generative Prompting,Segmentation Prompting,8.0,The code processes 2D image bounding boxes and...,This code defines data pipelines for 3D object...
3,https://github.com/HasinthakaPiyumal/AI-Patter...,LLM based Multimodal Generative Prompting: 0.5955,None,None,1.0,The code snippet is a dataset class for 3D obj...,This code defines a PyTorch Dataset for 3D obj...
4,https://github.com/HasinthakaPiyumal/AI-Patter...,LLM based Multimodal Generative Prompting: 0.5...,None,None,1.0,The code implements a data loading and augment...,This code implements a specialized data loadin...


In [6]:
new_patterns = """
Model Abstraction Pattern -The Model Abstraction Pattern involves creating a unified interface (or abstraction layer) that decouples your application logic from the specific Large Language Model (LLM) implementation being used. This pattern allows developers to seamlessly swap out different LLMs—such as Gemini, GPT, or Llama—as business needs, costs, or performance requirements change. By using this layer, all interactions, including prompt formatting and response parsing, remain consistent, preventing vendor lock-in and simplifying maintenance. This flexibility ensures your implementation can easily evolve and utilize the best available model for a given task.
ClassicalModels -Classical Models, in this context, refer to machine learning and deep learning architectures that are typically trained from scratch or on general datasets, distinguishing them from the specialized practice of fine-tuning large pre-trained language models (LLMs). These models include a wide range of algorithms such as logistic regression, Support Vector Machines (SVMs), decision trees, and foundational deep learning structures like Convolutional Neural Networks (CNNs) and Recurrent Neural Networks (RNNs). They are often applied to structured data, image classification, or natural language tasks where the model must learn features and representations directly from the given training set. This category represents the vast landscape of traditional and deep learning methodologies applied before the widespread adoption of fine-tuning multi-purpose foundational models.
Preprocessing Text and Numerical data - The Preprocessing Text and Numerical Data Pattern involves a critical series of transformations applied to raw input data before it is fed into machine learning models. For numerical data, this often includes tasks like scaling (e.g., Min-Max or standardization), handling missing values through imputation, and feature engineering. For text data, this involves tokenization, stop word removal, stemming/lemmatization, and converting words into numerical vector representations like Word Embeddings or TF-IDF scores.
"""

In [18]:
predicted_clusters

,code file,predicted labels,llm predicted high level pattern,llm predicted l2 pattern,confidence score,gemini verification explaination,code summary,new_pattern_match
0,https://github.com/HasinthakaPiyumal/AI-Patter...,LLM based Multimodal Generative Prompting: 0.4868,LLM based Multimodal Generative Prompting,3D Prompting,9.0,The code processes multimodal data (Lidar poin...,This code implements a data pipeline for 3D ob...,NaN
1,https://github.com/HasinthakaPiyumal/AI-Patter...,LLM based Multimodal Generative Prompting: 0.2496,None,None,1.0,The code implements 3D object pose estimation ...,This code implements a 3D object pose estimati...,Preprocessing Text and Numerical data
2,https://github.com/HasinthakaPiyumal/AI-Patter...,LLM based Multimodal Generative Prompting: 0.5620,LLM based Multimodal Generative Prompting,Segmentation Prompting,8.0,The code processes 2D image bounding boxes and...,This code defines data pipelines for 3D object...,NaN
3,https://github.com/HasinthakaPiyumal/AI-Patter...,LLM based Multimodal Generative Prompting: 0.5955,None,None,1.0,The code snippet is a dataset class for 3D obj...,This code defines a PyTorch Dataset for 3D obj...,None
4,https://github.com/HasinthakaPiyumal/AI-Patter...,LLM based Multimodal Generative Prompting: 0.5...,None,None,1.0,The code implements a data loading and augment...,This code implements a specialized data loadin...,Preprocessing Text and Numerical data
...,...,...,...,...,...,...,...,...
1536,https://github.com/HasinthakaPiyumal/AI-Patter...,Cross-lingual LLM Prompting: 0.3539,None,None,1.0,The code snippet performs file system operatio...,This code implements a robust pattern for mana...,NaN
1537,https://github.com/HasinthakaPiyumal/AI-Patter...,LLM based Multimodal Generative Prompting: 0.5130,LLM based Multimodal Generative Prompting,Multimodal Interaction Augmentation,9.0,The code extracts a YouTube video's speech tra...,This code exemplifies an AI pattern focused on...,NaN
1538,https://github.com/HasinthakaPiyumal/AI-Patter...,Tool Use for LLMs: 0.8034,Tool Use for LLMs,Structured Action Space for Agentic LLM,9.0,"The code defines a finite, structured set of c...",The code implements an **AI Agent Control** pa...,NaN
1539,https://github.com/HasinthakaPiyumal/AI-Patter...,Tool Use for LLMs: 0.6991,None,None,1.0,The code snippet contains generic utility func...,The provided code primarily consists of utilit...,NaN


In [19]:
# Build LLM helper to map None rows to one of the proposed new patterns
from langchain_core.prompts import ChatPromptTemplate
from typing import Optional
if not os.environ.get("GOOGLE_API_KEY"):
    os.environ["GOOGLE_API_KEY"] = getpass.getpass("Enter API key for Google Gemini: ")
    
llm = init_chat_model("gemini-2.5-flash", model_provider="google_genai", temperature=0)
selection_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a precise classifier. Given a code summary and a list of candidate patterns, pick the single best matching pattern name or reply None if no good match. look for exact matches."),
    ("human", "Candidates:\n{candidates}\n\nCode summary:\n{code_summary}\n\nReply with only one pattern name from the list or None. Look for exact matches. If no exact match, reply None."),
])

def pick_new_pattern(code_summary: str, candidates: str) -> str:
    msg = selection_prompt.format_messages(candidates=candidates, code_summary=code_summary)
    resp = llm.invoke(msg)
    return resp.content.strip()
# Apply to rows where high-level pattern is missing
predicted_clusters["new_pattern_match"] = np.nan
for idx, row in tqdm(predicted_clusters.iterrows()):
    if row['llm predicted high level pattern']!='None':
        continue
    code_summary = row.get("code summary", "")
    predicted_clusters.at[idx, "new_pattern_match"] = pick_new_pattern(code_summary, new_patterns)

0it [00:00, ?it/s]/tmp/ipykernel_136580/2358927700.py:23: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'Preprocessing Text and Numerical data' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  predicted_clusters.at[idx, "new_pattern_match"] = pick_new_pattern(code_summary, new_patterns)
2it [00:10,  5.04s/it]/tmp/ipykernel_136580/2358927700.py:23: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'Preprocessing Text and Numerical data' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  predicted_clusters.at[idx, "new_pattern_match"] = pick_new_pattern(code_summary, new_patterns)
1541it [1:07:36,  2.63s/it]
1541it [1:07:36,  2.63s/it]


In [20]:
# Save dataframe with new_pattern_match column
output_path = "/home/hasinthaka/Documents/Projects/AI/Pattern Mining/pipeline/notebooks/result/pattern_classification_verification_results_with_new_patterns.json"
predicted_clusters.to_json(output_path, orient="records", force_ascii=False)
output_path

'/home/hasinthaka/Documents/Projects/AI/Pattern Mining/pipeline/notebooks/result/pattern_classification_verification_results_with_new_patterns.json'